# Lab 21 solution, notebook 1: roll the die while `quick` is down, and price the two replies

The worked answer to the challenge of `21a_meet_litellm.ipynb`. The setup repeats the notebook's names, its dice
tool and its Router with a fallback, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
import random

import litellm
from litellm import Router, completion_cost

litellm.request_timeout = 60
litellm.suppress_debug_info = True

NAMES = ["Biscuit", "Pixel", "Mango", "Captain Whiskers", "Noodle", "Zorro"]
numbered = ", ".join(f"{number}. {name}" for number, name in enumerate(NAMES, start=1))


def roll_dice(sides: int) -> int:
    """Roll one die and return the number it shows."""
    print(f"🎲 TOOL CALLED: roll_dice({sides})")
    return random.randint(1, sides)


tools = [
    {
        "type": "function",
        "function": {
            "name": "roll_dice",
            "description": "Roll one die and return the number it shows.",
            "parameters": {
                "type": "object",
                "properties": {"sides": {"type": "integer", "description": "How many sides the die has"}},
                "required": ["sides"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

The Router of Part G: `quick` points, on purpose, at an address where nothing answers, and `careful` is its
fallback.

In [3]:
NOWHERE = "http://127.0.0.1:9/v1"

broken_list = [
    {
        "model_name": "quick",
        "litellm_params": {"model": "openai/" + settings["MODEL_COMPARE"], "api_base": NOWHERE},
        "model_info": {"id": "quick-on-openai"},
    },
    {
        "model_name": "careful",
        "litellm_params": {"model": "openai/" + MODEL, "reasoning_effort": EFFORT},
        "model_info": {"id": "careful-on-openai"},
    },
]

fallback_router = Router(model_list=broken_list, fallbacks=[{"quick": ["careful"]}])

## Task 1: the tool round through the Router while `quick` is down

The two calls of Part E, with `fallback_router.completion(model="quick", ...)` where Part E had
`litellm.completion(model="openai/" + MODEL, ...)`. The reasoning effort is no longer in the call, because the
deployment carries it. After each reply we print the id of the deployment that wrote it.

In [4]:
dice_messages = [{"role": "user", "content": f"My six name ideas: {numbered}. Roll a die to choose one, then tell me my kitten's name."}]

first = fallback_router.completion(model="quick", messages=dice_messages, tools=tools)

message = first.choices[0].message
print("tool calls:", [(call.function.name, call.function.arguments) for call in message.tool_calls or []])
print("deployment:", first._hidden_params["model_id"])

tool calls: [('roll_dice', '{"sides":6}')]
deployment: careful-on-openai


In [5]:
dice_messages.append(message.model_dump(exclude_none=True))

for call in message.tool_calls:
    arguments = json.loads(call.function.arguments)
    result = roll_dice(**arguments)
    print("the die shows", result, "which is", NAMES[result - 1])
    dice_messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

final = fallback_router.completion(model="quick", messages=dice_messages, tools=tools)

print(final.choices[0].message.content)
print("deployment:", final._hidden_params["model_id"])

🎲 TOOL CALLED: roll_dice(6)
the die shows 6 which is Zorro


The die rolled a 6, so your kitten’s name is **Zorro**!
deployment: careful-on-openai


One `TOOL CALLED` line. The die showed 6, which is Zorro in our list, and the reply names Zorro. Both replies print `deployment: careful-on-openai`.

The outage changed nothing in the tool round. The request for the tool and the final answer were both written by the
fallback, and `roll_dice` ran in our code, as it does whichever deployment answers. Each of the two calls waited a
few seconds first, while the Router retried `quick`.

## Task 2: what the two replies cost

`completion_cost` for each of the two replies, then the same sum by hand: the token counts of both replies times
the prices in `models.env`, divided by a million.

In [6]:
estimate = completion_cost(completion_response=first) + completion_cost(completion_response=final)

PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])

input_tokens = first.usage.prompt_tokens + final.usage.prompt_tokens
output_tokens = first.usage.completion_tokens + final.usage.completion_tokens
by_hand = (input_tokens * PRICE_IN + output_tokens * PRICE_OUT) / 1_000_000

print("input tokens: ", input_tokens)
print("output tokens:", output_tokens)
print(f"completion_cost: ${estimate:.8f}")
print(f"by hand:         ${by_hand:.8f}")

input tokens:  420
output tokens: 40
completion_cost: $0.00006200
by hand:         $0.00006200


The two sums agree: \$0.00006200 for 420 input tokens and 40 output tokens over the two replies. By hand that is (420 × 0.10 + 40 × 0.50) / 1,000,000. The second call sent the whole conversation again, with the model's request and the die's result, which is why two short replies add up to 420 input tokens.

What the estimate cannot contain: the attempts on `quick` that failed. `completion_cost` prices a reply, and a
failed attempt has none. Here those attempts cost nothing, because nothing answered at that address. A deployment
that fails after it has started to answer (a timeout in the middle of a long reply, for example) can be billed by
the vendor and still leave no reply to price. The vendor's invoice is the only place such a cost shows.